# Риск незачёта - стартовый ноутбук

Задача: по таблице студентов предсказать, кто **не получит зачёт** по предмету,
и назвать **3 фактора**, которые влияют сильнее всего.

Разделитель `;`, английские названия колонок, 649 строк, 33 колонки.

Ноутбук запускается целиком и даёт baseline. Ваша задача - побить его и объяснить результат.
Места для работы помечены **TODO**.

## 0. Загрузка

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, RocCurveDisplay)
from sklearn.inspection import permutation_importance

pd.set_option('display.max_columns', 100)

# ВАЖНО: разделитель - точка с запятой, а не запятая
df = pd.read_csv('student-por.csv', sep=';')
print(df.shape)
df.head()

## 1. Целевая переменная

В файле нет готовой колонки «зачёт / незачёт» - её нужно сделать самим.

`G3` - итоговый балл по 20-балльной шкале. Порог зачёта в португальской системе - **10 баллов**.
Всё, что ниже, считаем незачётом.

In [ ]:
df['no_pass'] = (df['G3'] < 10).astype(int)

print(df['no_pass'].value_counts())
print('\nдоля незачётов:', round(df['no_pass'].mean(), 3))

Классы **несбалансированы**: незачётов всего 15 %.

Модель, которая всем подряд ставит «зачёт», получит accuracy 85 % и не найдёт ни одного
студента в зоне риска. Поэтому дальше - ROC-AUC и PR-AUC, а не accuracy.

In [ ]:
# У 15 студентов G3 == 0. Это не «написал на ноль» - скорее человек не дошёл до итога.
print('G3 == 0 :', (df['G3'] == 0).sum(), 'студентов')
df[df['G3'] == 0][['absences', 'studytime', 'failures', 'G1', 'G2', 'G3']].head()

**TODO - решите и обоснуйте:** оставляете этих 15 студентов в выборке или убираете?
У них при этом бывают ненулевые `G1` и `G2` - то есть человек учился, а потом пропал.
Это отдельный сюжет: «бросил» и «не сдал» - это одно и то же событие или разные?

## 2. Смотрим на данные

Расшифровка колонок - в условии задачи. Ниже - русские подписи, чтобы графики читались.

In [ ]:
RU = {
    'absences': 'пропуски занятий', 'studytime': 'часы самоподготовки', 'failures': 'прошлые незачёты',
    'G1': 'контрольная 1', 'G2': 'контрольная 2', 'G3': 'итоговый балл',
    'traveltime': 'дорога до школы', 'famrel': 'отношения в семье', 'freetime': 'свободное время',
    'goout': 'встречи с друзьями', 'Dalc': 'алкоголь в будни', 'Walc': 'алкоголь в выходные',
    'health': 'здоровье', 'age': 'возраст', 'Medu': 'образование матери', 'Fedu': 'образование отца',
    'schoolsup': 'доп. поддержка школы', 'famsup': 'поддержка семьи', 'paid': 'платные занятия',
    'higher': 'хочет учиться дальше', 'internet': 'интернет дома', 'romantic': 'отношения',
    'activities': 'кружки', 'nursery': 'ходил в садик', 'school': 'учебное заведение',
    'sex': 'пол', 'address': 'город/село', 'famsize': 'размер семьи', 'Pstatus': 'родители вместе',
    'Mjob': 'работа матери', 'Fjob': 'работа отца', 'reason': 'причина выбора', 'guardian': 'опекун',
}

In [ ]:
key = ['absences', 'studytime', 'failures', 'G1', 'G2', 'goout', 'Dalc']
df.groupby('no_pass')[key].mean().round(2)

In [ ]:
# TODO: постройте свои графики - распределения, boxplot, матрицу корреляций.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col in zip(axes, ['absences', 'G2', 'studytime']):
    df[df.no_pass == 0][col].plot(kind='hist', bins=15, alpha=.6, ax=ax, label='зачёт')
    df[df.no_pass == 1][col].plot(kind='hist', bins=15, alpha=.6, ax=ax, label='незачёт')
    ax.set_title(RU.get(col, col)); ax.legend()
plt.tight_layout(); plt.show()

## 3. Готовим признаки

Главная ловушка этого датасета: **`G3` нельзя оставлять в признаках**. Целевая переменная
сделана прямо из него, так что модель с `G3` внутри покажет ROC-AUC около 1.0 и будет
абсолютно бесполезной. Это утечка, и жюри проверяет её в первую очередь.

Текстовые колонки (`sex`, `Mjob`, `higher`, ...) превращаем в числа через one-hot.

In [ ]:
X = df.drop(columns=['G3', 'no_pass'])
y = df['no_pass']

X = pd.get_dummies(X, drop_first=True)
print(X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(len(X_train), 'обучение /', len(X_test), 'тест')

## 4. Baseline

`class_weight='balanced'` заставляет модель не игнорировать редкий класс.

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
rf = RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42)

for name, model in [('логрегрессия', logreg), ('случайный лес', rf)]:
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    print(f'{name:15s} ROC-AUC {roc_auc_score(y_test, p):.3f}   PR-AUC {average_precision_score(y_test, p):.3f}')

In [ ]:
# Одна оценка на одном сплите - это шум. Проверяем на кросс-валидации.
cv = StratifiedKFold(5, shuffle=True, random_state=42)
for name, model in [('логрегрессия', logreg), ('случайный лес', rf)]:
    s = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    print(f'{name:15s} ROC-AUC {s.mean():.3f} +- {s.std():.3f}')

## 5. Порог и матрица ошибок

Модель выдаёт вероятность, а решение «звать студента на разговор или нет» требует **порога**.
0.5 - не закон природы. Что дороже: пропустить студента в зоне риска или зря побеспокоить того,
у кого всё в порядке?

In [ ]:
proba = rf.predict_proba(X_test)[:, 1]
threshold = 0.5   # TODO: подберите осмысленный порог и обоснуйте его

pred = (proba >= threshold).astype(int)
print(confusion_matrix(y_test, pred))
print()
print(classification_report(y_test, pred, target_names=['зачёт', 'незачёт']))

RocCurveDisplay.from_predictions(y_test, proba)
plt.show()

## 6. Три главных фактора

`feature_importances_` у леса смещена в сторону признаков с большим числом уникальных значений.
Честнее — **permutation importance**: перемешиваем одну колонку и смотрим, насколько просела метрика.

Считать её надо на **отложенной выборке**. Если посчитать на обучающей, лес, который её запомнил,
покажет схлопнувшиеся важности: всё, кроме G1 и G2, уйдёт ровно в ноль, а сами G1/G2 просядут
почти в десять раз — и вы недооцените все остальные факторы.

In [ ]:
r = permutation_importance(rf, X_test, y_test, n_repeats=30, random_state=42, scoring='roc_auc')
imp = pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False)

print('ТОП-3 фактора:')
for i, (name, val) in enumerate(imp.head(3).items(), 1):
    print(f'  {i}. {name:20s} падение ROC-AUC при перемешивании: {val:.4f}')

top = imp.head(10)[::-1]
top.index = [RU.get(n, n) for n in top.index]
top.plot(kind='barh', figsize=(7, 4))
plt.title('Permutation importance'); plt.tight_layout(); plt.show()

In [ ]:
# Важность говорит НАСКОЛЬКО фактор влияет, но не В КАКУЮ СТОРОНУ.
# Данные стандартизованы, поэтому коэффициенты логрегрессии можно сравнивать между собой.
coefs = pd.Series(logreg.named_steps['logisticregression'].coef_[0], index=X.columns).sort_values()
print('Сильнее всего СНИЖАЮТ риск незачёта:'); print(coefs.head(5).round(3))
print('\nСильнее всего ПОВЫШАЮТ риск незачёта:'); print(coefs.tail(5).round(3))

## 7. Уровень 2: прогноз до первой контрольной

`G1` и `G2` - это уже оценки по предмету, и они забирают почти всю предсказательную силу.
Модель с ними хорошо работает к концу семестра, когда что-то менять поздно.

Уберите их - и получите модель, полезную куратору **в начале** семестра. Метрика упадёт,
и это нормально. А вот тройка главных факторов станет совсем другой - вот это и есть интересный результат.

In [ ]:
X2 = X.drop(columns=['G1', 'G2'])
s = cross_val_score(rf, X2, y, cv=cv, scoring='roc_auc')
print(f'без G1 и G2:  ROC-AUC {s.mean():.3f} +- {s.std():.3f}')

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=.25, stratify=y, random_state=42)
rf2 = RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42).fit(X2_tr, y2_tr)
r2 = permutation_importance(rf2, X2_te, y2_te, n_repeats=30, random_state=42, scoring='roc_auc')
print()
print(pd.Series(r2.importances_mean, index=X2.columns).sort_values(ascending=False).head(5).round(4).to_string())

## 8. Что дальше - TODO

1. **Проверьте устойчивость топ-3.** Меняется ли тройка от `random_state`, от модели, от метрики?
   Если да - какая тройка честная и как вы это докажете?
2. **Сделайте свои признаки.** Динамика между контрольными (`G2 - G1`), пропуски на час
   самоподготовки, суммарный алкоголь (`Dalc + Walc`), «есть ли хоть одна поддержка».
3. **Проверьте перенос.** Рядом лежит `student-mat.csv` - те же 33 колонки, но другой предмет
   (математика, 395 студентов, доля незачётов вдвое выше: 32,9 % против 15,4 %). Обучите модель
   **без G1 и G2** на португальском, проверьте на математике. Модель со всеми признаками почти
   не просядет - оценки по предмету предсказывают оценки по предмету. А вот без них качество
   рухнет. Что это говорит про ваши «три главных фактора»?
4. **Объясните человеку.** Финальный вывод должен быть понятен куратору группы, который
   не знает слова «градиентный бустинг».
5. **Подумайте о последствиях.** Что делать со студентом, которого модель отнесла к группе риска?
   Что будет, если она ошиблась?

И держите в голове: корреляция - не причинность. Если признак хорошо предсказывает незачёт,
это ещё не значит, что, воздействуя на него, вы измените результат.